# 11 · Prompt tuning — trainable "soft tokens" at the input

**Recap (09):** the input is a **sequence of token-embedding vectors**. **Prompt
tuning** prepends a few **trainable** vectors to that sequence — fake "tokens" that
aren't real words, just learnable steering vectors. The entire model is frozen; only
these prepended vectors train.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## The mechanism: glue learnable vectors onto the front

Real input is a `(n_tokens, d)` sequence. We create `P` extra vectors — the **soft
prompt** — and stick them at the front. The frozen model then reads them as if they
were extra context at the start.

In [ ]:
d = 4
seq = rng.normal(size=(3, d))          # the real input: 3 token vectors

P = 2                                   # number of soft-prompt vectors
soft_prompt = rng.normal(size=(P, d))   # the ONLY trainable thing
new_seq = np.vstack([soft_prompt, seq]) # prepend them

print("real input :", seq.shape[0], "tokens")
print("after       :", new_seq.shape[0], "tokens  (2 soft + 3 real)")
print("trainable params: just the soft prompt =", soft_prompt.size)

## How tiny is it?

The trainable count is `num_virtual_tokens × d` — independent of how big the model
is. For 20 virtual tokens at `d = 1536`:

In [ ]:
num_virtual_tokens = 20
d_model = 1536
trainable = num_virtual_tokens * d_model
print(f"prompt-tuning trainable params: {trainable:,}  ({trainable/1.5e9:.4%} of the model)")

## Recap & the catch

- **Prompt tuning** prepends `P` trainable vectors to the input sequence; everything
  else is frozen.
- Trainable params = `P × d` only — the smallest of the additive methods.
- **The catch:** the new knobs sit *only at the input* (layer 0). The frozen layers
  above must do all the work of interpreting them, so prompt tuning often
  **underperforms at smaller models** (like our 1.5B) — expect this in the results.

That weakness motivates the next method, which adds knobs at **every** layer.

**BAM!** Next: **`12 — prefix tuning`.**